# Stochastic nets

Giving each transition an exponential firing `rate` turns the reachability graph into a continuous-time Markov chain (CTMC). `steady_state` solves `pi Q = 0, sum(pi) = 1`. We use an M/M/1/K queue: `arrive` (rate lambda) moves a token from `Free` to `Queue`, `serve` (rate mu) moves it back.

```mermaid
flowchart LR
    Free((Free K)) --> arrive[arrive] --> Queue((Queue)) --> serve[serve] --> Free
```

Analytically, `pi_k` is proportional to `(lambda/mu)^k`, `k = 0..K`.

In [1]:
from petrilab.petri.stochastic import expected_tokens, mm1k, steady_state, throughput

n = mm1k(1.0, 2.0, 3)
pi = steady_state(n)
for m, p in pi.items():
    print(n.marking_dict(m), round(p, 4))
print("sum:", round(sum(pi.values()), 6))

{'Free': 3, 'Queue': 0} 0.5333
{'Free': 2, 'Queue': 1} 0.2667
{'Free': 1, 'Queue': 2} 0.1333
{'Free': 0, 'Queue': 3} 0.0667
sum: 1.0


With lambda=1, mu=2, K=3 the distribution over queue length 0..3 is 8/15, 4/15, 2/15, 1/15, exactly the analytic M/M/1/K result (each step halves the probability).

In [2]:
print([round(x, 4) for x in (8 / 15, 4 / 15, 2 / 15, 1 / 15)])
print([round(pi[n.marking(Free=3 - k, Queue=k)], 4) for k in range(4)])

[0.5333, 0.2667, 0.1333, 0.0667]
[0.5333, 0.2667, 0.1333, 0.0667]


Comparing numerically: the solver matches the closed form.

In [3]:
print(
    "E[queue]:",
    round(expected_tokens(n, pi, "Queue"), 4),
    "expected",
    round((4 + 4 + 3) / 15, 4),
)
print("throughput arrive:", round(throughput(n, pi, "arrive"), 4))
print("throughput serve: ", round(throughput(n, pi, "serve"), 4))

E[queue]: 0.7333 expected 0.7333
throughput arrive: 0.9333
throughput serve:  0.9333


The expected queue length is 11/15. Throughput is rate times the probability the transition is enabled; accepted arrivals equal completed services in steady state, as flow balance requires. Arrivals are lost when the queue is full, which is why throughput is below lambda.